# 4차시 ― 머신러닝 직접 만들기 · 붓꽃 품종 분류

> 2차시에서는 Hugging Face `pipeline` 으로 이미 만들어진 모델을 **가져다 써** 보았습니다. 이번 노트북에서는 scikit-learn 으로 분류 모델을 **직접 학습**시켜 봅니다. 데이터를 준비하고, 모델을 학습시키고, 성능을 평가하는 머신러닝의 전체 흐름을 한 번에 경험합니다.
>
> 짝꿍 노트북 **s4-04(집값 예측)** 가 숫자 값을 맞히는 **회귀**였다면, 이 노트북은 **어느 품종인지** 맞히는 **분류**입니다.
>
> **실행 방법**: 코드 셀을 클릭하고 `Shift + Enter`. 위에서 아래로 순서대로 실행하세요. 앞 셀에서 만든 변수를 뒤 셀에서 사용합니다.

**다루는 내용**
1. 분류란 무엇인가 (회귀와의 차이)
2. 데이터 불러오기 · DataFrame 으로 탐색
3. 입력(X) · 정답(y) 나누기, 학습/평가 분리
4. 모델 3종 학습 · 비교
5. 평가 — 정확도 · 혼동행렬 · classification_report
6. 왜 그렇게 분류했나 — 결정 트리 · 특성 중요도
7. 새 데이터 예측해 보기
8. (도전) 타이타닉 생존 분류 맛보기
9. 정리 · 다음 차시 예고

## 1. 분류란 무엇인가

**머신러닝**은 사람이 규칙을 일일이 적는 대신, **데이터로부터 규칙(패턴)을 스스로 배우게** 하는 방법입니다. 그중에서도 정답이 있는 데이터로 배우는 방식을 **지도학습**이라 하고, 지도학습은 다시 두 가지로 나뉩니다.

| 구분 | 무엇을 맞히나 | 정답의 형태 | 예시 |
|---|---|---|---|
| **회귀** | 연속된 **숫자 값** | 실수 (예: 320,500,000원) | 집값 예측 (s4-04) |
| **분류** | 정해진 **범주(클래스)** | 라벨 (예: A·B·C 중 하나) | 붓꽃 품종 · 스팸/정상 메일 |

이번 노트북의 문제는 분류입니다. 꽃잎·꽃받침의 길이와 너비를 보고 **붓꽃이 세 품종 중 어느 것인지** 맞힙니다. 정답이 "320,500,000" 같은 숫자가 아니라 "setosa / versicolor / virginica" 중 하나라는 점이 회귀와의 가장 큰 차이입니다.

## 2. 라이브러리 불러오기와 데이터 준비

분류에 필요한 라이브러리를 먼저 불러옵니다. 이번 데이터는 scikit-learn 안에 **내장**되어 있어 파일을 따로 내려받지 않아도 됩니다.

In [ ]:
# scikit-learn(sklearn): 파이썬의 대표 머신러닝 라이브러리입니다.
# Colab 에는 아래 라이브러리가 대부분 이미 설치되어 있습니다.
# 만약 설치가 안 되어 있다면 아래 주석(#)을 지우고 한 번 실행하세요.
# !pip install -q scikit-learn pandas matplotlib seaborn

import numpy as np                 # 숫자 계산 (약어: np)
import pandas as pd                # 표(DataFrame) 형태 데이터 다루기 (약어: pd)
import matplotlib.pyplot as plt    # 그래프 그리기 (약어: plt)
import seaborn as sns              # 그래프를 더 보기 좋게 (약어: sns)

import warnings
warnings.filterwarnings('ignore')  # 실행에 영향 없는 경고 메시지를 숨깁니다.

print("라이브러리 준비 완료")

### 2.1 붓꽃 데이터 불러오기

`load_iris()` 는 붓꽃 데이터셋을 돌려줍니다. 통계학에서 아주 유명한 예제 데이터로, 붓꽃 150송이를 측정한 표입니다.

- **입력(특성)**: 꽃받침 길이·너비, 꽃잎 길이·너비 (총 4개, 단위 cm)
- **정답(품종)**: setosa · versicolor · virginica (3종, 각 50송이)

In [ ]:
from sklearn.datasets import load_iris

iris = load_iris()   # 붓꽃 데이터를 통째로 담은 객체

# iris 안에 무엇이 들어 있는지 키 목록을 봅니다.
print(iris.keys())
# 결과 예: dict_keys(['data', 'target', 'frame', 'target_names', ...])

print("특성 이름:", iris.feature_names)   # 입력 4개의 이름
print("품종 이름:", iris.target_names)    # 정답 3개의 이름
print("데이터 크기:", iris.data.shape)    # (150, 4) = 150송이 x 4개 특성

### 2.2 표(DataFrame)로 바꿔서 보기

`iris.data` 는 숫자만 든 배열이라 한눈에 보기 어렵습니다. 3차시에서 배운 pandas `DataFrame` 으로 바꾸면 엑셀 표처럼 컬럼 이름과 함께 볼 수 있습니다.

In [ ]:
# 1) 입력 데이터(4개 특성)를 DataFrame 으로 만듭니다.
df = pd.DataFrame(iris.data, columns=iris.feature_names)

# 2) 정답(품종)도 컬럼으로 추가합니다.
#    iris.target 은 0,1,2 숫자입니다. 사람이 읽기 쉽게 품종 이름도 함께 넣습니다.
df['target'] = iris.target                              # 0 / 1 / 2 (숫자 라벨)
df['species'] = iris.target_names[iris.target]          # setosa / versicolor / virginica

# .head(): 처음 5개 행을 미리 봅니다.
df.head()

In [ ]:
# .describe(): 수치형 컬럼의 기초 통계량 (개수·평균·표준편차·최소/최대 등)
df.describe()

In [ ]:
# .value_counts(): 각 품종이 몇 개씩 있는지 셉니다.
# 세 품종이 50개씩 고르게 들어 있어, 한쪽으로 치우치지 않은 균형 잡힌 데이터입니다.
print(df['species'].value_counts())

### 2.3 그림으로 살펴보기

숫자만 봐서는 품종이 잘 나뉘는지 알기 어렵습니다. **산점도**로 두 특성을 양 축에 놓고 품종별로 색을 다르게 칠해 보면, 품종이 공간에서 어떻게 떨어져 있는지 한눈에 보입니다.

In [ ]:
# 꽃잎(petal) 길이 vs 너비 산점도. 품종(species)별로 색을 다르게 표시합니다.
plt.figure(figsize=(7, 5))
sns.scatterplot(
    data=df,
    x='petal length (cm)',   # 가로축: 꽃잎 길이
    y='petal width (cm)',    # 세로축: 꽃잎 너비
    hue='species',           # 품종별 색 구분
    s=60                     # 점 크기
)
plt.title('Petal length vs width by species')
plt.show()

# 해석: setosa(왼쪽 아래)는 다른 두 품종과 확실히 떨어져 있어 쉽게 구분됩니다.
#       versicolor 와 virginica 는 일부 겹쳐서, 이 둘을 가르는 게 분류의 핵심입니다.

In [ ]:
# pairplot: 4개 특성을 둘씩 모든 조합으로 산점도를 그려 줍니다(특성 간 관계 한눈에 보기).
# 대각선은 각 특성의 분포(히스토그램)입니다. (그리는 데 몇 초 걸릴 수 있습니다.)
sns.pairplot(df, vars=iris.feature_names, hue='species', height=1.8)
plt.show()

# 해석: 꽃잎(petal) 관련 두 특성에서 품종이 가장 잘 나뉩니다.
#       즉 꽃잎 정보가 품종을 구분하는 데 특히 중요하다는 힌트를 얻을 수 있습니다.

### 연습문제 1
위 산점도는 **꽃잎(petal)** 으로 그렸습니다. 같은 방식으로 **꽃받침(sepal)** 의 길이(`sepal length (cm)`)와 너비(`sepal width (cm)`)로 산점도를 그려 보세요. 꽃잎 때보다 품종이 더 잘 나뉘는지, 덜 나뉘는지 눈으로 확인해 봅니다.

In [ ]:
# 아래에 작성하세요
# 힌트: 위 scatterplot 코드에서 x, y 만 sepal 컬럼으로 바꾸면 됩니다.



## 3. 입력(X) · 정답(y) 나누기, 학습/평가 분리

머신러닝 모델에게는 **입력과 정답을 따로** 건네줍니다.

- **X (입력/특성)**: 예측에 사용할 자료 — 여기서는 4개 측정값
- **y (정답/라벨)**: 맞혀야 하는 답 — 여기서는 품종(0·1·2)

In [ ]:
# X: 입력으로 쓸 4개 특성만 모읍니다. (species, target 같은 정답 컬럼은 제외)
X = iris.data            # 모양 (150, 4)
# y: 정답(품종). 숫자 라벨 0/1/2 를 사용합니다.
y = iris.target          # 모양 (150,)

print("X 크기:", X.shape)   # (150, 4)
print("y 크기:", y.shape)   # (150,)
print("y 앞 10개:", y[:10]) # [0 0 0 0 0 0 0 0 0 0]  (앞쪽은 모두 setosa)

### 3.1 왜 학습용과 평가용으로 나누나

가진 데이터를 **전부** 학습에 쓰면, 모델이 그 데이터의 답을 외워 버려 점수가 좋아 보일 수 있습니다. 하지만 정작 **처음 보는 새 데이터**에서는 틀릴 수 있는데, 이걸 미리 알 길이 없습니다(이를 과적합이라 합니다).

그래서 데이터를 미리 둘로 나눕니다. **학습용(train)** 으로만 공부시키고, 떼어 둔 **평가용(test)** 으로 처음 보는 문제처럼 점수를 매깁니다. `train_test_split()` 이 이 작업을 한 줄로 해 줍니다.

In [ ]:
from sklearn.model_selection import train_test_split

# train_test_split: 데이터를 랜덤하게 섞은 뒤 학습용/평가용으로 나눕니다.
#   - test_size=0.2 : 전체의 20%를 평가용으로 (나머지 80%는 학습용)
#   - random_state=42 : 난수를 고정해, 실행할 때마다 같은 분할이 나오게 합니다.
#   - stratify=y : 세 품종의 비율을 학습/평가 양쪽에 똑같이 유지합니다.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("학습용 X:", X_train.shape)   # (120, 4)  = 150 의 80%
print("평가용 X:", X_test.shape)    # (30, 4)   = 150 의 20%
print("학습용 정답 분포:", np.bincount(y_train))  # 품종별 개수 (40,40,40)
print("평가용 정답 분포:", np.bincount(y_test))   # 품종별 개수 (10,10,10)

## 4. 모델 학습하고 비교하기

이제 분류 모델을 만들어 학습시킵니다. scikit-learn 모델은 사용법이 거의 똑같아서, 어떤 모델이든 다음 두 단계로 씁니다.

1. `모델.fit(X_train, y_train)` — 학습용 데이터로 패턴을 **배웁니다**.
2. `모델.predict(X_test)` — 배운 것으로 새 데이터의 답을 **예측합니다**.

성격이 다른 분류기 세 가지를 같은 방식으로 학습시켜 비교합니다.

| 모델 | 한 줄 설명 |
|---|---|
| **LogisticRegression** | 이름은 회귀지만 **분류** 모델. 특성을 가중합해 클래스 확률을 계산합니다. |
| **DecisionTreeClassifier** | "꽃잎 길이가 2.5 보다 크면…" 같은 **예/아니오 질문**을 잇따라 던져 나눕니다. |
| **KNeighborsClassifier** | 새 꽃과 **가장 가까운 이웃 k개**의 품종을 보고 다수결로 정합니다. |

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

# 세 모델을 딕셔너리에 담아 둡니다. {모델이름: 모델객체}
models = {
    "LogisticRegression": LogisticRegression(max_iter=200),
    "DecisionTree":       DecisionTreeClassifier(random_state=42),
    "KNN":                KNeighborsClassifier(n_neighbors=5),
}

results = {}  # 모델별 정확도를 저장할 빈 딕셔너리

for name, model in models.items():        # 모델을 하나씩 꺼내서
    model.fit(X_train, y_train)           # 1) 학습용으로 학습
    y_pred = model.predict(X_test)        # 2) 평가용으로 예측
    acc = accuracy_score(y_test, y_pred)  # 3) 정답과 비교해 정확도 계산
    results[name] = acc
    print(f"{name:20s} 정확도: {acc:.4f}")  # 예: 0.9667 → 96.67%

# 정확도(accuracy): 전체 중 맞힌 비율. 분류에서 가장 기본이 되는 점수입니다.

In [ ]:
# 세 모델의 정확도를 막대그래프로 비교합니다.
plt.figure(figsize=(6, 4))
names = list(results.keys())
accs  = list(results.values())
bars = plt.bar(names, accs, color=['#7c6cff', '#9aa0aa', '#9aa0aa'])
plt.ylim(0.8, 1.0)
plt.ylabel('Accuracy')
plt.title('Model accuracy comparison')
for bar, a in zip(bars, accs):            # 막대 위에 숫자 표시
    plt.text(bar.get_x() + bar.get_width()/2, a + 0.003, f"{a:.3f}", ha='center')
plt.show()

# 붓꽃은 쉬운 데이터라 세 모델 모두 90%대 후반으로 잘 맞힙니다.
# 보통은 이렇게 여러 모델을 비교한 뒤 가장 좋은 것을 고릅니다.

### 연습문제 2
`KNeighborsClassifier` 의 `n_neighbors`(참고할 이웃 수)를 5 대신 **1** 과 **15** 로 바꿔 각각 학습하고, 평가용 정확도를 출력해 보세요. 이웃 수에 따라 점수가 어떻게 달라지나요?

In [ ]:
# 아래에 작성하세요
# 힌트:
# knn1 = KNeighborsClassifier(n_neighbors=1)
# knn1.fit(X_train, y_train)
# print("k=1 :", accuracy_score(y_test, knn1.predict(X_test)))



## 5. 더 자세히 평가하기 — 혼동행렬과 리포트

정확도 하나만 보면 **어떤 품종을 어떤 품종으로** 헷갈렸는지 알 수 없습니다. **혼동행렬(confusion matrix)** 은 "실제 품종" 대 "예측 품종" 을 표로 만들어, 어디서 틀렸는지 보여 줍니다.

여기서는 위 비교에서 잘 나온 **LogisticRegression** 으로 자세히 살펴봅니다.

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

# 평가에 쓸 모델을 다시 학습시킵니다.
clf = LogisticRegression(max_iter=200)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

# 혼동행렬: 행=실제 품종, 열=예측 품종
cm = confusion_matrix(y_test, y_pred)
print(cm)
# 대각선(왼위→오른아래)에 있는 수가 '맞게 예측한 개수'입니다.
# 대각선 밖의 0 이 아닌 수가 '틀린(헷갈린) 경우'입니다.

In [ ]:
# 혼동행렬을 히트맵으로 그리면 어디서 틀렸는지 한눈에 보입니다.
plt.figure(figsize=(5.5, 4.5))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Purples',
    xticklabels=iris.target_names,   # 열: 예측 품종
    yticklabels=iris.target_names    # 행: 실제 품종
)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

# setosa 는 완벽히 맞히고, 겹치던 versicolor/virginica 에서 한두 개 틀리는 정도입니다.

### classification_report 읽는 법

`classification_report` 는 품종마다 점수를 더 자세히 보여 줍니다.

- **precision(정밀도)**: 그 품종이라고 **예측한 것 중** 실제로 맞은 비율
- **recall(재현율)**: 그 품종인 **실제 것 중** 모델이 찾아낸 비율
- **f1-score**: precision 과 recall 의 균형 점수 (높을수록 좋음)

In [ ]:
# target_names 를 넘기면 0/1/2 대신 품종 이름으로 보여 줍니다.
print(classification_report(y_test, y_pred, target_names=iris.target_names))

## 6. 왜 그렇게 분류했을까 — 결정 트리 들여다보기

머신러닝 모델은 종종 "왜 그렇게 판단했는지" 가 보이지 않습니다. 그런데 **결정 트리** 는 사람이 읽을 수 있는 **예/아니오 질문의 흐름도**라서, 어떤 기준으로 나눴는지 그림으로 볼 수 있습니다.

In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree

# 그림으로 보기 좋게, 깊이를 3으로 제한한 작은 트리를 학습합니다.
tree = DecisionTreeClassifier(max_depth=3, random_state=42)
tree.fit(X_train, y_train)

plt.figure(figsize=(13, 7))
plot_tree(
    tree,
    feature_names=iris.feature_names,   # 질문에 쓰인 특성 이름
    class_names=iris.target_names,      # 예측 품종 이름
    filled=True,                        # 품종별로 색칠
    rounded=True, fontsize=9
)
plt.title('Decision Tree (max_depth=3)')
plt.show()

# 읽는 법: 맨 위 질문(예: 'petal length <= 2.45?')부터 시작해,
#          맞으면 왼쪽 / 아니면 오른쪽으로 내려가며 품종을 좁혀 갑니다.
#          맨 아래 칸(leaf)의 class 가 최종 예측 품종입니다.

### 6.1 특성 중요도 — 어떤 측정값이 결정적이었나

트리가 품종을 나눌 때 어떤 특성을 많이 활용했는지 점수로 나타낸 것이 **특성 중요도(feature importance)** 입니다. 앞서 산점도에서 "꽃잎이 잘 나눈다" 고 본 직관을 숫자로 확인합니다.

In [ ]:
# .feature_importances_ : 각 특성이 분류에 기여한 정도(합이 1)
importances = tree.feature_importances_

# 보기 좋게 큰 순서로 정렬해 가로 막대로 그립니다.
order = np.argsort(importances)           # 작은→큰 순서의 인덱스
plt.figure(figsize=(7, 4))
plt.barh(np.array(iris.feature_names)[order], importances[order], color='#7c6cff')
plt.xlabel('Importance')
plt.title('Feature importance (Decision Tree)')
plt.tight_layout()
plt.show()

# 꽃잎(petal) 길이·너비의 중요도가 압도적으로 높게 나옵니다.
# 즉 이 트리는 주로 '꽃잎'을 보고 품종을 나누고 있다는 뜻입니다.

## 7. 새 데이터로 직접 예측해 보기

학습이 끝난 모델은 **새로 측정한 꽃** 의 품종도 예측할 수 있습니다. 측정값 4개를 넣어 `predict` 하면 됩니다. 입력 순서는 학습 때와 같아야 합니다 — `[꽃받침 길이, 꽃받침 너비, 꽃잎 길이, 꽃잎 너비]`.

In [ ]:
# 새로 측정한 꽃 1송이 (단위 cm). 2차원 형태 [[...]] 로 넣는 점에 주의하세요.
new_flower = [[5.1, 3.5, 1.4, 0.2]]   # 꽃잎이 짧고 가늚 → setosa 같아 보이는 값

pred = clf.predict(new_flower)              # 품종 번호(0/1/2) 예측
print("예측 품종 번호:", pred[0])
print("예측 품종 이름:", iris.target_names[pred[0]])

# predict_proba: 각 품종일 '확률'도 볼 수 있습니다.
proba = clf.predict_proba(new_flower)[0]
for name, p in zip(iris.target_names, proba):
    print(f"  {name:12s}: {p*100:5.1f}%")

In [ ]:
# 꽃잎이 큰 값을 넣어 보면 다른 품종으로 예측됩니다.
new_flower2 = [[6.7, 3.0, 5.2, 2.3]]   # 꽃잎이 길고 넓음 → virginica 같은 값
pred2 = clf.predict(new_flower2)
print("예측 품종:", iris.target_names[pred2[0]])

### 연습문제 3
꽃받침·꽃잎 측정값 4개를 **직접 정해** `my_flower` 변수에 넣고, 모델이 어떤 품종으로 예측하는지 확인해 보세요. (예: `[[6.0, 2.7, 4.5, 1.5]]`) 산점도를 떠올리며 versicolor 가 나올 만한 값을 넣어 보면 좋습니다.

In [ ]:
# 아래에 작성하세요
# my_flower = [[??, ??, ??, ??]]
# print(iris.target_names[clf.predict(my_flower)[0]])



### 연습문제 4
`train_test_split` 의 `test_size` 를 **0.5**(절반을 평가용으로)로 바꿔 데이터를 다시 나누고, `LogisticRegression` 을 학습시켜 평가용 정확도를 출력해 보세요. 학습 데이터가 줄면 점수가 어떻게 변하는지 관찰합니다. (`random_state=42`, `stratify=y` 는 그대로 두세요.)

In [ ]:
# 아래에 작성하세요
# 힌트:
# X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.5, random_state=42, stratify=y)
# m = LogisticRegression(max_iter=200); m.fit(X_tr, y_tr)
# print(accuracy_score(y_te, m.predict(X_te)))



### 연습문제 5
4장에서 만든 `models` 딕셔너리에 **새 모델 하나**를 더 넣어 함께 비교해 보세요. 예를 들어 다음 둘 중 하나를 추가하면 됩니다.

- `from sklearn.svm import SVC` 의 `SVC()`
- `from sklearn.ensemble import RandomForestClassifier` 의 `RandomForestClassifier(random_state=42)`

추가한 모델까지 포함해 정확도를 출력합니다.

In [ ]:
# 아래에 작성하세요
# 힌트: 위 import 후, models["SVC"] = SVC() 처럼 한 줄 추가하고 4장의 for 문을 다시 실행합니다.



## 8. (도전 · 선택) 타이타닉 생존 분류 맛보기

붓꽃은 "세 품종 중 하나" 를 고르는 분류였습니다. 분류에는 **둘 중 하나**(예/아니오)를 맞히는 문제도 많은데, 이를 **이진 분류** 라 합니다. 대표 예제가 타이타닉 승객의 **생존 여부(생존/사망)** 예측입니다.

여기서는 seaborn 에 내장된 타이타닉 데이터로, 앞에서 배운 흐름(X·y 나누기 → 학습/평가 분리 → fit/predict → 정확도)이 **그대로** 적용된다는 점만 가볍게 확인합니다. 시간이 없으면 이 절은 건너뛰어도 됩니다.

In [ ]:
# seaborn 내장 타이타닉 데이터 불러오기 (인터넷이 막혀 있으면 실패할 수 있습니다)
titanic = sns.load_dataset('titanic')
print("데이터 크기:", titanic.shape)
titanic[['survived', 'pclass', 'sex', 'age', 'fare']].head()
# survived: 1=생존, 0=사망 (이게 맞혀야 할 정답 y)
# pclass(객실 등급), sex(성별), age(나이), fare(요금) 등을 입력으로 씁니다.

In [ ]:
# 간단히 4개 특성만 사용합니다.
cols = ['pclass', 'sex', 'age', 'fare']
data = titanic[cols + ['survived']].dropna()   # 결측값 있는 행은 제거(맛보기라 간단히)

# 모델은 숫자만 이해하므로, 글자 'male/female' 을 숫자로 바꿉니다.
data['sex'] = data['sex'].map({'male': 0, 'female': 1})

X_t = data[cols]          # 입력 4개
y_t = data['survived']    # 정답 (생존 1 / 사망 0)

X_tr, X_te, y_tr, y_te = train_test_split(
    X_t, y_t, test_size=0.2, random_state=42, stratify=y_t
)
print("학습용:", X_tr.shape, "/ 평가용:", X_te.shape)

In [ ]:
# 붓꽃 때와 똑같이 fit → predict → accuracy 로 평가합니다.
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(random_state=42)
model.fit(X_tr, y_tr)
y_p = model.predict(X_te)
print("타이타닉 생존 예측 정확도:", round(accuracy_score(y_te, y_p), 4))

# 어떤 특성이 생존 예측에 중요했는지도 같은 방법으로 볼 수 있습니다.
for name, imp in sorted(zip(cols, model.feature_importances_), key=lambda x: -x[1]):
    print(f"  {name:8s}: {imp:.3f}")
# 보통 성별(sex)과 객실 등급(pclass)·요금(fare)이 생존에 크게 작용한 것으로 나옵니다.

### 연습문제 6
위 타이타닉 코드에서 `RandomForestClassifier` 대신 **`LogisticRegression(max_iter=1000)`** 으로 바꿔 학습하고, 평가용 정확도를 출력해 보세요. (앞서 만든 `X_tr, X_te, y_tr, y_te` 를 그대로 사용하면 됩니다.)

In [ ]:
# 아래에 작성하세요



## 9. 정리

- **분류**는 정답이 "범주(품종·생존 여부)" 인 문제로, 숫자 값을 맞히는 **회귀**(s4-04 집값)와 짝을 이룹니다.
- 모든 모델은 **`fit`(학습) → `predict`(예측)** 로 똑같이 쓰며, scikit-learn 덕분에 모델을 바꿔 끼우기 쉽습니다.
- 성능은 **정확도** 하나로 끝내지 말고, **혼동행렬·classification_report** 로 "어디서 틀렸는지" 까지 살핍니다.
- **결정 트리·특성 중요도** 로 "왜 그렇게 분류했는지" 를 들여다볼 수 있습니다.
- 학습한 모델은 **새 데이터** 의 품종을 예측할 수 있습니다.

이번 4차시에서는 **머신러닝** 으로 분류기를 직접 학습시켰습니다. 다음 **5차시** 에서는 같은 "분류" 문제를 **딥러닝(BERT)** 으로 풀어, 문장의 감정을 분류하고 RAG 챗봇까지 만들어 봅니다. 직접 만든 머신러닝 모델과 딥러닝 모델이 어떻게 다른지 비교해 보세요.